# Ceteris paribus — technical companion

Technical companion to `cp_walkthrough.ipynb`. No teaching figures: only
commented code, prints and small tables.

What the lecture asserts, this notebook measures:

- §2 does the answer depend on the grid we chose?
- §3 the staircase, and how much of it is the seed
- §4 the impossible fraction across all 30 features, not the six on the slide
- §5 why the distance criterion fails, derived and then measured
- §6 a ceteris paribus slope and a LIME coefficient are the same quantity

In [1]:
%pip install -q scikit-learn matplotlib numpy lime

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import pearsonr, spearmanr

RANDOM_STATE = 42

data = load_breast_cancer()
X_all, y_all = data.data, data.target
feature_names = list(data.feature_names)
feat_idx = {f: i for i, f in enumerate(feature_names)}

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

instance_idx = 67
row = X_test[instance_idx]
train_mean, train_std = X_train.mean(axis=0), X_train.std(axis=0)
P = len(feature_names)


def cp_profile(base_row, j, grid):
    probe = np.tile(base_row, (len(grid), 1))
    probe[:, j] = grid
    return model.predict_proba(probe)[:, 1]


def feature_grid(base_row, j, span=2.5, n=200):
    lo = max(X_all[:, j].min(), base_row[j] - span * train_std[j])
    hi = min(X_all[:, j].max(), base_row[j] + span * train_std[j])
    return np.linspace(lo, hi, n)


print(f"setup matches module 03: {len(X_train)} train / {len(X_test)} test, "
      f"patient #{instance_idx} at P(benign) = {proba_test[instance_idx]:.4f}")

setup matches module 03: 426 train / 143 test, patient #67 at P(benign) = 0.5812


## 2 · Does the grid decide the answer?

A ceteris paribus plot has two free parameters nobody reports: how far the
sweep goes, and how many points it uses. The lecture uses ±2.5σ and 200
points. If the conclusion moves when those move, it is a fact about the grid.

In [3]:
J = feat_idx["worst perimeter"]
R = feat_idx["worst radius"]

# the walkthrough's geometric envelope, recomputed here so each span can be read
# against it: with her radius frozen, this is the only interval the perimeter can
# occupy without implying a tumour no real patient has
ratio_real = X_all[:, J] / X_all[:, R]
poss_lo, poss_hi = ratio_real.min() * row[R], ratio_real.max() * row[R]

steps_25, imp_by_span = {}, {}
print(f"{'span':>6} {'n':>5} {'swing in P':>11} {'largest step':>13} {'step at':>9} {'impossible':>11}")
for span in (1.0, 2.5, 5.0):
    for n in (50, 200, 800):
        g = feature_grid(row, J, span=span, n=n)
        p = cp_profile(row, J, g)
        st = np.abs(np.diff(p))
        imp = (~((g >= poss_lo) & (g <= poss_hi))).mean()
        if span == 2.5:
            steps_25[n] = st.max()
        if n == 200:
            imp_by_span[span] = imp
        print(f"{span:>6.1f} {n:>5} {p.max() - p.min():>11.3f} "
              f"{st.max():>13.3f} {g[np.argmax(st)]:>9.1f} {imp:>10.0%}")

g0 = feature_grid(row, J, span=2.5, n=200)
k0 = int(np.argmax(np.abs(np.diff(cp_profile(row, J, g0)))))
print(f"\n'step at' is the LEFT EDGE of the grid interval the jump crosses: in the")
print(f"lecture's row (span 2.5, n = 200) the step runs {g0[k0]:.1f} -> {g0[k0 + 1]:.1f}.")

print("\nThe swing is the same to three decimals in all nine settings — even at")
print("+/-1 sigma, because the profile has already flattened by then. Widening")
print("the sweep adds nothing but impossible rows: from +/-1 sigma to the +/-2.5")
print(f"the lecture uses, the swing does not move and the impossible fraction goes")
print(f"{imp_by_span[1.0]:.0%} -> {imp_by_span[2.5]:.0%} ({imp_by_span[5.0]:.0%} at +/-5 sigma). The extra width buys no signal.")

print("\nThe largest single step, however, shrinks as the grid gets finer. At one")
print(f"fixed span (+/-2.5 sigma) it reads {steps_25[50]:.3f} at n=50, {steps_25[200]:.3f} at n=200 and")
print(f"{steps_25[800]:.3f} at n=800. A step height is an artefact of the spacing: a finer")
print("grid cuts the same jump into more pieces. Quote the swing, which is a")
print("property of the model. Never quote a step height.")

  span     n  swing in P  largest step   step at  impossible
   1.0    50       0.212         0.048     112.4        64%
   1.0   200       0.212         0.041     112.6        64%
   1.0   800       0.212         0.015     112.7        63%
   2.5    50       0.212         0.094     113.1        84%
   2.5   200       0.212         0.039     115.1        84%
   2.5   800       0.212         0.023     112.6        83%
   5.0    50       0.212         0.126     111.9        88%
   5.0   200       0.212         0.046     115.0        88%
   5.0   800       0.212         0.025     115.2        88%

'step at' is the LEFT EDGE of the grid interval the jump crosses: in the
lecture's row (span 2.5, n = 200) the step runs 115.1 -> 115.8.

The swing is the same to three decimals in all nine settings — even at
+/-1 sigma, because the profile has already flattened by then. Widening
the sweep adds nothing but impossible rows: from +/-1 sigma to the +/-2.5
the lecture uses, the swing does not move a

## 3 · The staircase, and how much of it is the seed

The steps are where the forest changes its mind. But a forest is a random
object: refit it with a different seed and the splits move. Module 03 made
the same check on the decision boundary and found it wanders between 0.06σ
and 0.48σ from the patient. Here it is for the profile itself.

In [4]:
base_g = feature_grid(row, J)
base_p = cp_profile(row, J, base_g)
swings, biggest_at, curves = [], [], []
for s in range(12):
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=s)
    m.fit(X_train, y_train)
    probe = np.tile(row, (len(base_g), 1))
    probe[:, J] = base_g
    p = m.predict_proba(probe)[:, 1]
    curves.append(p)
    swings.append(p.max() - p.min())
    biggest_at.append(base_g[np.argmax(np.abs(np.diff(p)))])
curves = np.array(curves)

print(f"over 12 forest seeds, sweeping the same feature on the same patient:")
print(f"  swing in P:          median {np.median(swings):.3f}   "
      f"min {min(swings):.3f}   max {max(swings):.3f}")
print(f"  the biggest step is at worst perimeter = {np.median(biggest_at):.1f} "
      f"(median), spread {min(biggest_at):.1f} to {max(biggest_at):.1f} (left edges)")
print(f"  pointwise spread of P across seeds: median sd {np.median(curves.std(axis=0)):.3f}, "
      f"max {curves.std(axis=0).max():.3f}")
lecture_swing = base_p.max() - base_p.min()
print(f"\nthis notebook's own forest is seed {RANDOM_STATE}, and its swing is "
      f"{lecture_swing:.3f} —")
print(f"  above all twelve refits (max {max(swings):.3f}, median {np.median(swings):.3f}). The headline")
print("  number of the lecture is an upper-tail draw of the seed, not a typical")
print("  forest. Quote it with that caveat, or quote the twelve-seed median.")

print("\nThe shape is stable; the exact location of a step is not. Read the profile")
print("as a direction and a rough magnitude, never as 'the threshold is at 115.1'.")

over 12 forest seeds, sweeping the same feature on the same patient:
  swing in P:          median 0.161   min 0.116   max 0.181
  the biggest step is at worst perimeter = 114.3 (median), spread 112.1 to 115.1 (left edges)
  pointwise spread of P across seeds: median sd 0.015, max 0.024

this notebook's own forest is seed 42, and its swing is 0.212 —
  above all twelve refits (max 0.181, median 0.161). The headline
  number of the lecture is an upper-tail draw of the seed, not a typical
  forest. Quote it with that caveat, or quote the twelve-seed median.

The shape is stable; the exact location of a step is not. Read the profile
as a direction and a rough magnitude, never as 'the threshold is at 115.1'.


## 4 · The impossible fraction, all 30 features

The lecture shows six. Here is every feature, using the same rule: freeze the
most-correlated partner, sweep the feature, and ask which grid points imply a
ratio no real patient has.

Only features with a strong partner get a meaningful test — with an
uncorrelated partner the ratio constraint is vacuous. So the correlation is
reported next to the fraction.

In [5]:
C = np.corrcoef(X_train.T)
rows_out = []
for j in range(P):
    c = np.abs(C[j].copy())
    c[j] = 0
    k = int(np.argmax(c))
    if row[k] == 0:
        continue
    r = X_all[:, j] / np.where(X_all[:, k] == 0, np.nan, X_all[:, k])
    lo, hi = np.nanmin(r), np.nanmax(r)
    g = feature_grid(row, j)
    ok = (g >= lo * row[k]) & (g <= hi * row[k])
    # how TIGHT the observed ratio is, as a pure number: a dimensionless shape
    # factor stays in a narrow band, a ratio with units does not
    band = hi / lo if lo > 0 else np.inf
    rows_out.append((feature_names[j], feature_names[k], c[k], (~ok).mean(), band))

rows_out.sort(key=lambda t: -t[2])
print(f"{'feature':<26} {'strongest partner':<26} {'|r|':>5} {'impossible':>11} {'band hi/lo':>11}")
for name, pname, corr, frac, band in rows_out[:12]:
    print(f"{name:<26} {pname:<26} {corr:>5.2f} {frac:>10.0%} {band:>11.2f}")

strong = [f for _, _, c, f, _ in rows_out if c > 0.9]
allf = [f for _, _, _, f, _ in rows_out]
print(f"\nover all {len(rows_out)} features:            median {np.median(allf):.0%} impossible")
print(f"over the {len(strong)} with a partner at |r| > 0.9: median {np.median(strong):.0%}")

corrs = np.array([c for _, _, c, _, _ in rows_out])
fracs = np.array([f for _, _, _, f, _ in rows_out])
bands = np.array([b for _, _, _, _, b in rows_out])
print(f"\nrank correlation with the impossible fraction, over these {len(rows_out)} features:")
print(f"  |r| with the frozen partner:  Spearman {spearmanr(corrs, fracs).statistic:+.2f}"
      f"   (Pearson {pearsonr(corrs, fracs).statistic:+.2f})")
print(f"  width of the observed ratio band: Spearman {spearmanr(bands, fracs).statistic:+.2f}")

look = {name: (corr, frac, band) for name, _, corr, frac, band in rows_out}
print("\nBoth are tendencies; neither is the mechanism. The table refutes the easy")
print("story that the stronger the correlation the more of the curve is fiction:")
for name in ("mean radius", "mean area", "mean concavity"):
    corr, frac, band = look[name]
    print(f"  {name:<15} |r| {corr:.3f} -> {frac:>3.0%} impossible, ratio band {band:.2f} wide")
print(f"|r| falls by {look['mean radius'][0] - look['mean area'][0]:.3f} between the first two of those and the impossible")
print(f"fraction falls from {look['mean radius'][1]:.0%} to {look['mean area'][1]:.0%}. What separates them is the RATIO.")
print("perimeter/radius is dimensionless — a shape factor — and every real patient")
print(f"fits inside a band only {look['mean radius'][2]:.2f} wide, so freezing the partner pins the swept")
print("feature to a narrow interval. area/radius carries units of length, its band")
print(f"is {look['mean area'][2]:.2f} wide, and it pins almost nothing.")
print("\nSo the mechanism is the tightness of the constraint the frozen partner")
print("imposes, and a dimensionless ratio is what makes a constraint tight.")
print("Correlation is a proxy for that, and a leaky one. What stays true whatever")
print("the mechanism: ceteris paribus asks the model to hold fixed exactly the")
print("things that, in the data, are not free to stay fixed.")

feature                    strongest partner            |r|  impossible  band hi/lo
mean radius                mean perimeter              1.00        87%        1.15
mean perimeter             mean radius                 1.00        87%        1.15
worst radius               worst perimeter             0.99        84%        1.23
worst perimeter            worst radius                0.99        84%        1.23
mean area                  mean radius                 0.99        24%        4.44
worst area                 worst radius                0.99        23%        5.05
radius error               perimeter error             0.97        72%        4.88
perimeter error            radius error                0.97        46%        4.88
area error                 radius error                0.95        74%        4.85
mean concave points        mean concavity              0.92         6%       43.86
mean concavity             mean concave points         0.92         4%       43.86
mea

## 5 · Why the distance criterion cannot see it

The lecture reports that a nearest-neighbour test clears 100% of a sweep that
geometry rejects at 84%. That is not an implementation failure, and the
arithmetic is short enough to do here.

Two independent standardized rows differ on all $p$ coordinates, so their
squared distance is about $2p$ and their distance about $\sqrt{2p}$. A
ceteris paribus sweep moves **one** coordinate by at most a few σ, so it
travels a few σ — regardless of how absurd the row it lands on.

In [6]:
Z_all = (X_all - train_mean) / train_std
D = np.linalg.norm(Z_all[:, None, :] - Z_all[None, :, :], axis=2)
np.fill_diagonal(D, np.inf)
nn_real = D.min(axis=1)

probe = np.tile(row, (len(base_g), 1))
probe[:, J] = base_g
Zs = (probe - train_mean) / train_std
nn_sweep = np.linalg.norm(Zs[:, None, :] - Z_all[None, :, :], axis=2).min(axis=1)

print(f"predicted typical distance between two real patients: sqrt(2p) = {np.sqrt(2 * P):.2f}σ")
print(f"measured median pairwise distance:                     "
      f"{np.median(D[np.isfinite(D)]):.2f}σ")
print(f"\nnearest-neighbour distance among real patients: median {np.median(nn_real):.2f}σ, "
      f"95th pct {np.percentile(nn_real, 95):.2f}σ")
print(f"the sweep's distance to the nearest real patient: max {nn_sweep.max():.2f}σ")
print(f"  → exceeds the 95th percentile at {(nn_sweep > np.percentile(nn_real, 95)).mean():.0%} "
      f"of grid points")
print("\nThe sweep cannot get far enough to look unusual. A one-feature move is a")
print("small step in 30 dimensions even when it breaks a physical law, so any")
print("out-of-distribution check built on distance will pass it.")
print("\nStated as a rule: distance detects rows that are far from the data.")
print("Ceteris paribus produces rows that are CLOSE to the data and impossible.")
print("Those are different failures, and only the second one has a domain answer.")

predicted typical distance between two real patients: sqrt(2p) = 7.75σ
measured median pairwise distance:                     6.30σ

nearest-neighbour distance among real patients: median 2.07σ, 95th pct 4.45σ
the sweep's distance to the nearest real patient: max 2.50σ
  → exceeds the 95th percentile at 0% of grid points

The sweep cannot get far enough to look unusual. A one-feature move is a
small step in 30 dimensions even when it breaks a physical law, so any
out-of-distribution check built on distance will pass it.

Stated as a rule: distance detects rows that are far from the data.
Ceteris paribus produces rows that are CLOSE to the data and impossible.
Those are different failures, and only the second one has a domain answer.


## 6 · A ceteris paribus slope and a LIME coefficient

Module 03 §10b compares LIME's leading coefficient against a central finite
difference of the model, at steps of 0.1σ, 0.3σ and 1σ. Its result, after the
correction that module now carries: the **sign** of the leading coefficient
agrees with the finite difference for **97% / 100% / 100%** of the patients
whose leading feature actually moves. The direction does not depend on the
step. What depends on the step is the **ranking** — whether LIME's top feature
is the model's own goes 13% → 26% → 52%, and the cosine between the two
30-dimensional vectors 0.16 → 0.36 → 0.81.

That finite difference **is** a two-point ceteris paribus profile. So the two
modules measure the same object at different resolutions, and the two halves
of the object behave differently: the sign is stable across resolutions, the
magnitude and the ordering are not. LIME reports one number; ceteris paribus
draws the shape that number was read off.

Here it is directly — the CP slope across ±1σ against LIME's coefficient, on
the lecture's patient.

In [7]:
def cp_delta(base_row, j, h=1.0):
    """The model's raw change in P(benign) across ±h sigma of feature j — the
    numerator of the finite difference used in module 03 §10b."""
    a, b = base_row.copy(), base_row.copy()
    a[j] += h * train_std[j]
    b[j] -= h * train_std[j]
    return (model.predict_proba(a.reshape(1, -1))[0, 1]
            - model.predict_proba(b.reshape(1, -1))[0, 1])


def cp_slope(base_row, j, h=1.0):
    """Two-point ceteris paribus slope across ±h sigma, per sigma."""
    return cp_delta(base_row, j, h) / (2 * h)


e = LimeTabularExplainer(X_train, feature_names=feature_names,
                         discretize_continuous=False,
                         random_state=RANDOM_STATE).explain_instance(
    row, model.predict_proba, num_features=8, num_samples=5000, labels=(1,))
coef = dict(e.local_exp[1])

print(f"{'feature':<26} {'LIME coef':>10} {'CP slope ±1σ':>14} {'same sign?':>11}")
agree = flat = 0
for j, w in sorted(coef.items(), key=lambda t: -abs(t[1])):
    s = cp_slope(row, j)
    if s == 0:
        # A feature the forest is flat on has no direction to be right or wrong
        # about. np.sign(0) is 0, which equals neither +1 nor -1, so counting it
        # as a disagreement manufactures one — the trap module 03 fell into at
        # small steps. Here nothing is flat, but the guard is the point.
        flat += 1
        print(f"{feature_names[j]:<26} {w:>+10.4f} {s:>+14.4f} {'no move':>11}")
        continue
    ok = np.sign(w) == np.sign(s)
    agree += ok
    print(f"{feature_names[j]:<26} {w:>+10.4f} {s:>+14.4f} {str(bool(ok)):>11}")
print(f"\nsigns agree on {agree} of the {len(coef) - flat} selected features whose CP slope is "
      f"nonzero ({flat} of {len(coef)} are flat at ±1σ)")

lead = max(coef, key=lambda j: abs(coef[j]))
print(f"\nLIME's leading feature is '{feature_names[lead]}'. Its two-point reading "
      f"at four steps:")
table = [f"  {'h':>6} {'ΔP over ±h':>12} {'slope per σ':>13}"]
for h in (0.1, 0.3, 0.5, 1.0):
    table.append(f"  {h:>5.1f}σ {cp_delta(row, lead, h):>+12.4f} "
                 f"{cp_slope(row, lead, h):>+13.4f}")
print("\n".join(table))

d01, d10 = cp_delta(row, lead, 0.1), cp_delta(row, lead, 1.0)
s01, s10 = cp_slope(row, lead, 0.1), cp_slope(row, lead, 1.0)
print(f"\nThe slope falls {abs(s01 / s10):.1f}-fold between ±0.1σ and ±1σ, and that number is")
print("the one usually quoted. Most of it is the 1/h in the definition of a slope,")
print(f"not the model: what the model actually does, ΔP, only grows by {abs(d10 / d01):.2f}x over")
print(f"the same range ({d01:+.4f} to {d10:+.4f}), and it has already saturated by ±0.5σ.")
print("\nSo the honest version of module 03's finding is about normalisation, not")
print("about the model changing its mind: same sign, nearly the same jump, wildly")
print("different per-sigma rate. A coefficient hides the step it was measured at;")
print("a ceteris paribus profile shows it.")

feature                     LIME coef   CP slope ±1σ  same sign?
worst perimeter               -0.0709        -0.1062        True
worst concave points          -0.0457        -0.0482        True
worst texture                 -0.0213        -0.0394        True
area error                    -0.0184        -0.0265        True
mean texture                  -0.0153        -0.0639        True
mean perimeter                -0.0140        -0.0228        True
mean radius                   -0.0134        -0.0116        True
radius error                  -0.0089        -0.0183        True

signs agree on 8 of the 8 selected features whose CP slope is nonzero (0 of 8 are flat at ±1σ)

LIME's leading feature is 'worst perimeter'. Its two-point reading at four steps:
       h   ΔP over ±h   slope per σ
    0.1σ      -0.1506       -0.7530
    0.3σ      -0.1706       -0.2843
    0.5σ      -0.2119       -0.2119
    1.0σ      -0.2125       -0.1062



The slope falls 7.1-fold between ±0.1σ and ±1σ, and that number is
the one usually quoted. Most of it is the 1/h in the definition of a slope,
not the model: what the model actually does, ΔP, only grows by 1.41x over
the same range (-0.1506 to -0.2125), and it has already saturated by ±0.5σ.

So the honest version of module 03's finding is about normalisation, not
about the model changing its mind: same sign, nearly the same jump, wildly
different per-sigma rate. A coefficient hides the step it was measured at;
a ceteris paribus profile shows it.


## Summary

- The profile is a property of the model, not of the grid: resolution moves the
  step heights and nothing else, span moves only how much of the curve is
  impossible, and both are choices the analyst must report (§2).
- The staircase shape is stable across forest seeds; the position of any
  individual step is not, and this notebook's own seed sits above all twelve
  refits. Never quote a threshold off one of these plots, and treat the
  headline swing as one draw (§3).
- The impossible fraction is set by how tightly the frozen partner constrains
  the swept feature — a dimensionless ratio with a narrow observed band — and
  only loosely by the correlation between them (§4).
- A distance-based out-of-distribution check clears the whole sweep, because a
  one-feature move is small in 30 dimensions. Ceteris paribus rows are close
  to the data and impossible — a failure distance cannot detect (§5).
- A CP slope and a LIME coefficient are the same quantity at different
  resolutions. The sign survives the change of resolution; the per-sigma
  magnitude and the ranking are what module 03 measured moving (§6).

In [8]:
print("Lecture version, with the figures: cp_walkthrough.ipynb")

Lecture version, with the figures: cp_walkthrough.ipynb
